In [ ]:
from pathlib import Path

# Find the repository root so these notebooks work on other computers.
PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "Dataset Specs").is_dir() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "Dataset Specs").is_dir():
    raise FileNotFoundError("Start Jupyter from inside the Hospital Dataset project folder.")

import pandas as pd
import numpy as np
from pathlib import Path

data_dir = Path(
    f"{PROJECT_ROOT}/"
    "synthetic_hospital_outputs/90_10"
)

arima = pd.read_csv(
    data_dir / "arima_occupancy_predictions.csv",
    parse_dates=["date"]
)

rf = pd.read_csv(
    data_dir / "random_forest_occupancy_predictions.csv",
    parse_dates=["date"]
)

gb = pd.read_csv(
    data_dir / "gradient_boosting_occupancy_predictions.csv",
    parse_dates=["date"]
)

# Normalise dates so they match consistently
for frame in [arima, rf, gb]:
    frame["date"] = pd.to_datetime(frame["date"]).dt.normalize()

print("ARIMA columns:", arima.columns.tolist())
print("Random Forest columns:", rf.columns.tolist())
print("Gradient Boosting columns:", gb.columns.tolist())

print("Rows:", len(arima), len(rf), len(gb))

In [ ]:
# Keep only dates present in all three files
df_stack = arima.merge(
    rf, on="date", how="inner"
).merge(
    gb, on="date", how="inner",
    suffixes=("_arima", "_gb")
)

# Check actual occupancy values from ARIMA and Gradient Boosting
assert (
    df_stack["actual_occupied_beds_arima"]
    == df_stack["actual_occupied_beds_gb"]
).all(), "Actual occupancy values do not match!"

# Keep one actual occupancy column
df_stack = df_stack.rename(
    columns={
        "actual_occupied_beds_arima": "actual_occupied_beds"
    }
).drop(
    columns=["actual_occupied_beds_gb"]
)

print("Aligned rows:", len(df_stack))
print("Date range:", df_stack["date"].min(), "to", df_stack["date"].max())
print("Missing values:", df_stack.isna().sum().sum())
print(df_stack.columns.tolist())

In [ ]:
# Average the five Random Forest predictions
df_stack["rf_predicted_occupied_beds"] = df_stack[
    ["rf_1", "rf_2", "rf_3", "rf_4", "rf_5"]
].mean(axis=1)

# Average the five Gradient Boosting predictions
df_stack["gb_predicted_occupied_beds"] = df_stack[
    ["gb_1", "gb_2", "gb_3", "gb_4", "gb_5"]
].mean(axis=1)

# Keep only the columns needed for stacking
stacking_data = df_stack[
    [
        "date",
        "actual_occupied_beds",
        "arima_predicted_occupied_beds",
        "rf_predicted_occupied_beds",
        "gb_predicted_occupied_beds"
    ]
].copy()

print(stacking_data.head())
print("Rows:", len(stacking_data))

In [ ]:
X_meta = stacking_data[
    [
        "arima_predicted_occupied_beds",
        "rf_predicted_occupied_beds",
        "gb_predicted_occupied_beds"
    ]
].copy()

y_meta = stacking_data["actual_occupied_beds"].copy()

print("X_meta shape:", X_meta.shape)
print("y_meta shape:", y_meta.shape)

In [ ]:
from sklearn.model_selection import train_test_split

meta_split = int(len(X_meta) * 0.9)

X_meta_train = X_meta.iloc[:meta_split]
X_meta_test = X_meta.iloc[meta_split:]

y_meta_train = y_meta.iloc[:meta_split]
y_meta_test = y_meta.iloc[meta_split:]

print("Meta-training rows:", len(X_meta_train))
print("Meta-testing rows:", len(X_meta_test))
print(
    "Meta-training period:",
    stacking_data["date"].iloc[0],
    "to",
    stacking_data["date"].iloc[meta_split - 1]
)
print(
    "Meta-testing period:",
    stacking_data["date"].iloc[meta_split],
    "to",
    stacking_data["date"].iloc[-1]
)

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge

stacker = Pipeline([
    ("scaler", StandardScaler()),
    ("ridge", Ridge(alpha=1.0))
])

# Train only on the meta-training portion
stacker.fit(X_meta_train, y_meta_train)

# Predict the later, held-out portion
stack_pred = stacker.predict(X_meta_test)

print("Stacking model trained.")
print("Number of test predictions:", len(stack_pred))

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error

# Actual occupancy for the 15 held-out dates
y_true = y_meta_test.to_numpy()
y_pred = stack_pred

rmse = np.sqrt(mean_squared_error(y_true, y_pred))
mae = mean_absolute_error(y_true, y_pred)
mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100

print("Ridge Stacking Performance")
print(f"RMSE: {rmse:.3f} beds")
print(f"MAE:  {mae:.3f} beds")
print(f"MAPE: {mape:.3f}%")

In [ ]:
from pathlib import Path
import pandas as pd

output_dir = Path(
    f"{PROJECT_ROOT}/"
    "synthetic_hospital_outputs/90_10"
)
output_dir.mkdir(parents=True, exist_ok=True)

stacking_results = pd.DataFrame({
    "date": stacking_data["date"].iloc[meta_split:].to_numpy(),
    "actual_occupied_beds": y_true,
    "stacking_predicted_occupied_beds": y_pred
})

stacking_results.to_csv(
    output_dir / "ridge_stacking_predictions.csv",
    index=False
)

stacking_metrics = pd.DataFrame([{
    "model": "Ridge Stacking",
    "RMSE": rmse,
    "MAE": mae,
    "MAPE_percent": mape,
    "test_rows": len(y_true)
}])

stacking_metrics.to_csv(
    output_dir / "ridge_stacking_metrics.csv",
    index=False
)

print("Stacking predictions and metrics saved successfully.")

In [ ]:
print("Stacking test period:")
print(stacking_results["date"].min(), "to", stacking_results["date"].max())

print("Number of test days:", len(stacking_results))

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error
import numpy as np
import pandas as pd

comparison = stacking_results[
    ["date", "actual_occupied_beds"]
].copy()

# Load the saved baseline predictions
baseline = pd.read_csv(
    data_dir / "fusion_baseline_predictions.csv",
    parse_dates=["date"]
)

baseline["date"] = pd.to_datetime(baseline["date"]).dt.normalize()
comparison["date"] = pd.to_datetime(comparison["date"]).dt.normalize()

print("Baseline prediction columns:")
print(baseline.columns.tolist())

In [ ]:
# Match the baseline predictions to the stacking test dates
comparison = comparison.merge(
    baseline,
    on="date",
    how="inner",
    suffixes=("", "_baseline"),
    validate="one_to_one"
)

# Check that actual occupancy values match
assert (
    comparison["actual_occupied_beds"]
    == comparison["actual_occupied_beds_baseline"]
).all(), "Actual occupancy values do not match!"

# Add the Ridge Stacking predictions
comparison = comparison.merge(
    stacking_results[
        ["date", "stacking_predicted_occupied_beds"]
    ],
    on="date",
    how="inner",
    validate="one_to_one"
)

# Calculate performance metrics
actual = comparison["actual_occupied_beds"].to_numpy()

model_predictions = {
    "ARIMA": comparison["arima_predicted_occupied_beds"].to_numpy(),
    "Random Forest": comparison["rf_predicted_occupied_beds"].to_numpy(),
    "Gradient Boosting": comparison["gb_predicted_occupied_beds"].to_numpy(),
    "Simple Fusion Average": comparison["fusion_predicted_occupied_beds"].to_numpy(),
    "Ridge Stacking": comparison["stacking_predicted_occupied_beds"].to_numpy()
}

results = []

for model_name, predicted in model_predictions.items():
    results.append({
        "Model": model_name,
        "RMSE": np.sqrt(mean_squared_error(actual, predicted)),
        "MAE": mean_absolute_error(actual, predicted),
        "MAPE (%)": np.mean(np.abs((actual - predicted) / actual)) * 100
    })

comparison_metrics = pd.DataFrame(results).round(3)

print("Number of matched test days:", len(comparison))
display(comparison_metrics)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(15, 7))

plt.plot(
    comparison["date"],
    comparison["actual_occupied_beds"],
    label="Actual Occupancy",
    color="black",
    linewidth=2.5
)

plt.plot(
    comparison["date"],
    comparison["arima_predicted_occupied_beds"],
    label="ARIMA",
    alpha=0.8
)

plt.plot(
    comparison["date"],
    comparison["rf_predicted_occupied_beds"],
    label="Random Forest",
    alpha=0.8
)

plt.plot(
    comparison["date"],
    comparison["gb_predicted_occupied_beds"],
    label="Gradient Boosting",
    alpha=0.8
)

plt.plot(
    comparison["date"],
    comparison["fusion_predicted_occupied_beds"],
    label="Simple Fusion Average",
    alpha=0.8
)

plt.plot(
    comparison["date"],
    comparison["stacking_predicted_occupied_beds"],
    label="Ridge Stacking",
    linewidth=2
)

plt.title("Actual vs Predicted Hospital Bed Occupancy (132 Test Days)")
plt.xlabel("Date")
plt.ylabel("Occupied Beds")
plt.legend(loc="best")
plt.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()

plt.savefig(
    data_dir / "actual_vs_predictions_132_days.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
print(
    comparison[
        ["date", "actual_occupied_beds",
         "arima_predicted_occupied_beds"]
    ].head(10)
)

print(
    "\nNumber of unique ARIMA predictions:",
    comparison["arima_predicted_occupied_beds"].nunique()
)

print(
    "ARIMA prediction standard deviation:",
    comparison["arima_predicted_occupied_beds"].std()
)

In [ ]:
comparison_metrics.to_csv(
    data_dir / "model_comparison_15_days.csv",
    index=False
)

print("Model comparison table saved successfully.")